# AIaaS — Retrieval Quality Benchmark (MTEB)

The **Standard tier for the retrieval stack**: embeddings + **reranking** quality,
measured with the **Massive Text Embedding Benchmark (MTEB)** — the industry's
de-facto, leaderboard-comparable harness (the retrieval-world analog of MLPerf).

Unlike `embeddings_benchmark.ipynb` (which is a *throughput* proxy), this measures
**quality** on standardized tasks, so the scores line up with the public
[MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard):
- **Reranking** — MAP / MRR (e.g. AskUbuntuDupQuestions)
- **Semantic similarity (STS)** — Spearman correlation
- **Classification** — accuracy (logistic-regression probe on embeddings)

### Why this and not a homemade reranking timing
There's no MLPerf benchmark for embeddings/reranking — **MTEB is the standardized
harness everyone reports against**. Running it is what makes the number comparable;
a bespoke throughput test would be proxy-tier only. (Quality is GPU-independent;
we run on GPU just for speed.)

### Requirements
- Any CUDA GPU (or CPU — slower). Ungated model by default.
- The default 3-task set runs in minutes; the full MTEB suite is many hours.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "mteb", "sentence-transformers", "pandas"], check=False)
import mteb
print("mteb", getattr(mteb, "__version__", "?"))


## 2. Environment & hardware capture

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": (torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"),
    "vram_total_gb": (round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                      if DEVICE == "cuda" else None),
    "cuda": torch.version.cuda, "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__, "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
A small, representative task set (one reranking, one STS, one classification) so it
runs quickly. Browse more with `mteb.get_tasks(task_types=["Reranking"])`; the full
English suite is the leaderboard set (slow).

In [ ]:
MODEL = "BAAI/bge-small-en-v1.5"      # ungated; on the MTEB leaderboard
TASKS = [
    "AskUbuntuDupQuestions",          # Reranking  (MAP/MRR)
    "STSBenchmark",                   # STS        (Spearman)
    "Banking77Classification",        # Classification (accuracy)
]
# For the RAG-critical retrieval task type (heavier), add e.g. "SciFact".

OUT_DIR = "mteb_bench_results"
# Per-model raw dir so a re-run with a different MODEL can't contaminate the scrape.
RAW_DIR = os.path.join(OUT_DIR, "raw", MODEL.replace("/", "__"))
os.makedirs(RAW_DIR, exist_ok=True)
print(f"MODEL={MODEL}, device={DEVICE}, tasks={TASKS}")


## 4. Run MTEB
Resilient task loading (skips a name this MTEB version doesn't know), then evaluate.

In [ ]:
import time

model = mteb.get_model(MODEL)   # falls back to SentenceTransformer if not MTEB-native

tasks = []
for name in TASKS:
    try:
        got = mteb.get_tasks(tasks=[name])
    except Exception as e:
        print(f"skip task {name}: {e}"); got = []
    if not got:   # some mteb versions return [] (not raise) for an unknown name
        print(f"WARNING: task '{name}' resolved to nothing (renamed/unknown in this mteb version)")
    else:
        tasks += got
print("loaded", len(tasks), "tasks")
if not tasks:
    raise RuntimeError("No MTEB tasks resolved \u2014 check TASKS names, e.g. "
                       "mteb.get_tasks(task_types=['Reranking']).")

t0 = time.time()
try:
    results = mteb.MTEB(tasks=tasks).run(model, output_folder=RAW_DIR, verbosity=1)
except TypeError:
    # older/newer signature
    results = mteb.MTEB(tasks=tasks).run(model, output_folder=RAW_DIR)
elapsed = round(time.time() - t0, 1)
print(f"\nMTEB finished in {elapsed}s over {len(results)} task results")


## 5. Results — normalize, save, summarize

In [ ]:
import pandas as pd, glob

def get_name(r):
    n = getattr(r, "task_name", None)
    if n:
        return n
    t = getattr(r, "task", None)
    md_ = getattr(t, "metadata", None)
    return getattr(md_, "name", None) or str(n)

def get_type(r):
    t = getattr(r, "task", None)
    md_ = getattr(t, "metadata", None)
    return getattr(md_, "type", None)

def get_main(r):
    s = getattr(r, "main_score", None)
    if isinstance(s, (int, float)):
        return s
    if hasattr(r, "get_score"):
        try:
            return r.get_score()
        except Exception:
            pass
    sc = getattr(r, "scores", None)
    if isinstance(sc, dict):
        vals = []
        for _split, entries in sc.items():
            for e in (entries if isinstance(entries, list) else [entries]):
                if isinstance(e, dict) and isinstance(e.get("main_score"), (int, float)):
                    vals.append(e["main_score"])
        if vals:
            return sum(vals) / len(vals)
    return None

rows = []
for r in results:
    ms = get_main(r)
    rows.append({"task": get_name(r), "type": get_type(r),
                 "main_score": round(ms, 4) if isinstance(ms, (int, float)) else None})

# fallback: scrape any task JSONs MTEB wrote, if results objects were unhelpful
if not any(x["main_score"] is not None for x in rows):
    rows = []   # drop the all-None placeholders before scraping (avoid duplicate task rows)
    for fp in glob.glob(os.path.join(RAW_DIR, "**", "*.json"), recursive=True):
        try:
            d = json.load(open(fp))
            if "task_name" in d and "scores" in d:
                ms = None
                for _s, ent in d["scores"].items():
                    for e in (ent if isinstance(ent, list) else [ent]):
                        if isinstance(e, dict) and isinstance(e.get("main_score"), (int, float)):
                            ms = e["main_score"]
                rows.append({"task": d["task_name"], "type": d.get("task_type"),
                             "main_score": round(ms, 4) if ms is not None else None})
        except Exception:
            pass

# Average PER TASK TYPE only — MAP (reranking), Spearman (STS) and accuracy
# (classification) are different scales; a single blended mean is meaningless and
# the MTEB leaderboard never collapses them. Report per-type.
from collections import defaultdict
_by_type = defaultdict(list)
for x in rows:
    if isinstance(x["main_score"], (int, float)):
        _by_type[x["type"] or "unknown"].append(x["main_score"])
average_by_type = {k: round(sum(v) / len(v), 4) for k, v in _by_type.items()}
NORM = {
    "schema": "mteb-bench/1.0",
    "env": ENV, "model": MODEL, "device": DEVICE,
    "elapsed_s": elapsed, "results": rows,
    "average_by_type": average_by_type,
}
tag = (ENV["platform"] + "_" + MODEL).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"mteb_bench_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)
print(f"\n==== MTEB SUMMARY ====\n{MODEL}")
print("main_score by task type (metrics differ per type \u2014 not comparable across types):")
print(" ", average_by_type)
df = pd.DataFrame(rows)
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))


## 6. Reading the result

- **main_score** per task is what the MTEB leaderboard ranks on (MAP/MRR for
  reranking, Spearman for STS, accuracy for classification). Compare your model's
  numbers directly against the public [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).
- The scores are **GPU-independent** — they measure model *quality*, not hardware.
  Pair them with `embeddings_benchmark.ipynb` (throughput on *your* GPU) to choose
  an embedding/reranker model that is both **accurate enough** (MTEB) and **fast
  enough** (throughput) for your RAG service.
- For a leaderboard-grade submission, run the **full** MTEB English task set
  (`mteb.get_benchmark("MTEB(eng)")`) — many hours; the default trio here is the
  quick representative subset.